In [ ]:
# --- 코랩 준비: 드라이브 마운트 + catboost ---
import os, subprocess, sys
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")
try:
    import catboost  # noqa: F401
except ImportError:
    print("catboost 설치 중...", flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "catboost"], check=True)

# GPU 가드 — 없으면 즉시 중단. 조용히 CPU 로 몇 시간 태우는 사고를 막는다.
_r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                    capture_output=True, text=True)
if _r.returncode != 0:
    raise RuntimeError("GPU 가 없다. 런타임 > 런타임 유형 변경 > T4 GPU 로 바꿀 것.")
print("GPU:", _r.stdout.strip(), flush=True)


In [ ]:
# --- 학습 스크립트 풀기 (변형 네 개를 환경변수로 전환하는 단일 스크립트) ---
import base64, pathlib
_B64 = """..."""
_src = base64.b64decode("".join(_B64.split())).decode("utf-8")
pathlib.Path("/content/ablation.py").write_text(_src, encoding="utf-8")
compile(_src, "ablation.py", "exec")
print(f"ablation.py {len(_src):,}자, 문법 OK")


In [ ]:
# --- 변형을 순서대로 실행. 하나 끝날 때마다 즉시 드라이브에 저장한다 ---
# 세션이 중간에 끊겨도 그때까지 끝난 변형의 zip 은 드라이브에 남는다.
import os, shutil, subprocess, sys, time

RUNS = [
    ("r", {"AB_DROP_CAL": '[]', "AB_DECAY": "1.0", "AB_REST": "1", "AB_PB": "0"}),   # 휴식·등판밀도·파울,
    ("b", {"AB_DROP_CAL": '[]', "AB_DECAY": "1.0", "AB_REST": "0", "AB_PB": "1"}),   # cond_pb 맞대결,
    ("d", {"AB_DROP_CAL": '[]', "AB_DECAY": "0.25", "AB_REST": "0", "AB_PB": "0"}),   # cond_* 감쇠 0.25
]
OUT = "/content/drive/MyDrive/aimers_ablation"
os.makedirs(OUT, exist_ok=True)

for tag, env in RUNS:
    dst = f"{OUT}/submit_s1{tag}.zip"
    if os.path.exists(dst):
        print(f"[{tag}] 이미 있음 — 건너뜀", flush=True)
        continue
    wd = f"/content/run_{tag}"
    os.makedirs(wd, exist_ok=True)          # 변형마다 별도 CWD (model/ 이 섞이지 않게)
    e = dict(os.environ, **env)
    t0 = time.time()
    print(f"\n{'='*60}\n[{tag}] 시작  {env}\n{'='*60}", flush=True)
    log = f"{OUT}/log_s1{tag}.txt"
    with open(log, "w", encoding="utf-8") as lf:
        p = subprocess.Popen([sys.executable, "-u", "/content/ablation.py"],
                             cwd=wd, env=e, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                             errors="replace")
        for line in p.stdout:
            lf.write(line); lf.flush()
            if any(k in line for k in ("fold", "오프셋", "홀드아웃 환산", "Error",
                                       "Traceback", "통과", "생성 완료", "룩업", "휴식")):
                print(f"  [{tag}] {line.rstrip()}", flush=True)
        rc = p.wait()
    m = (time.time() - t0) / 60
    src = f"{wd}/submit_tuned.zip"
    if rc == 0 and os.path.exists(src):
        shutil.copy(src, dst)
        print(f"[{tag}] 완료 {m:.0f}분 -> {dst}  ({os.path.getsize(dst)/1e6:.0f}MB)", flush=True)
    else:
        print(f"[{tag}] 실패 rc={rc} ({m:.0f}분). 로그: {log}", flush=True)
    shutil.rmtree(wd, ignore_errors=True)   # 디스크 확보 (변형당 model/ 20MB + 중간 산출물)

print("\n전부 종료. 드라이브:", os.listdir(OUT), flush=True)
